[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/hubbard_ring/hubbard_ring.ipynb)

# The Hubbard Ring

This example computes the low-lying energy spectrum and spin dynamics of electrons hopping around a Hubbard ring at half filling, where two electrons sharing a site pay an on-site repulsion energy. Each site accommodates at most one spin-up and one spin-down electron, and hopping moves an electron to an adjacent site without flipping its spin. When on-site repulsion dominates hopping, charge motion freezes into brief virtual excursions that exchange neighbouring spins, so the low-energy spectrum converges to the antiferromagnetic Heisenberg ring and a single flipped spin propagates coherently to the opposite site.

A many-electron state across the spin orbitals is modelled as a 4-vector wedge such as `a ^ c ^ e ^ h`, which vanishes whenever two electrons occupy the same orbital and flips sign under fermion exchange, including hops across the periodic seam past the other three electrons. Any single-orbital operator lifts to the four-electron space as `(one_electron ^ (State & Complement)).trace(1, 3)`, where `State & Complement` removes each occupied orbital in turn, the wedge inserts its image with the fermion exchange sign, and the trace contracts the map's input against the removed orbital. Lifting the nearest-neighbour transfer map gives many-body hopping, while lifting each orbital projector `orbitals * (orbitals | Orbital)` gives its number operator, whose spin-up and spin-down composition on each site counts double occupancy, extending the two-electron [Hubbard dimer](../hubbard_dimer/hubbard_dimer.ipynb) to a closed ring.

In second-quantized Fock-space notation, the Hamiltonian reads $\hat H = -t \sum_{i=1}^{4} \sum_{\sigma \in \{\uparrow,\downarrow\}} (\hat c^\dagger_{i+1,\sigma} \hat c_{i\sigma} + \hat c^\dagger_{i\sigma} \hat c_{i+1,\sigma}) + U \sum_{i=1}^{4} \hat n_{i\uparrow} \hat n_{i\downarrow}$ with $\hat n_{i\sigma} = \hat c^\dagger_{i\sigma} \hat c_{i\sigma}$ and canonical anticommutators $\{\hat c_p, \hat c^\dagger_q\} = \delta_{pq}$, $\{\hat c_p, \hat c_q\} = 0$, where assembling matrix elements in an ordered occupation-bitstring basis $|n_1 \dots n_8\rangle = (\hat c^\dagger_1)^{n_1} \cdots (\hat c^\dagger_8)^{n_8} |0\rangle$ requires evaluating Jordan–Wigner parity strings $(-1)^{\sum_{r < p} n_r}$ across intermediate occupied orbitals on every hop, including boundary hops between sites $4$ and $1$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure, show_states
from examples.quantum.hubbard_ring import render

np.set_printoptions(precision=3, suppress=True)

# Eight orthogonal vector blades represent spin up and spin down on each of the four ring sites.
ga = Algebra("a+b+c+d+e+f+g+h+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                 # Scalar+
Orbital = ga.gatype.vector()                                                # Vector+
State = ga.gatype.k_vector(4)                                               # 4-vector+
Complement = ga.gatype.antivector()                                         # Antivector-
OneElectron = ga.gatype((Orbital, Orbital))                                 # Orbital+ <- Orbital+
Hamiltonian = ga.gatype((State, State))                                     # State+ <- State+

# Spin-up and spin-down orbitals on the four ring sites, and on each site's next neighbour around the ring.
orbitals = stack([stack([mv.a, mv.b]), stack([mv.c, mv.d]), stack([mv.e, mv.f]), stack([mv.g, mv.h])])   # [4, 2] Orbital+
next_orbitals = stack([orbitals[1], orbitals[2], orbitals[3], orbitals[0]])                              # [4, 2] Orbital+

## Lifting single-orbital maps to many-electron states

A linear map on single orbitals extends to a derivation on 4-vectors that applies the map to one occupied orbital at a time while leaving the other three in place and summing the four terms. Both nearest-neighbour hopping and each orbital's orthogonal projector lift through the same trace expression. On any site, composing the lifted spin-up projector with the lifted spin-down projector yields one when both spins occupy that site and zero otherwise, so summing those products around the ring counts doubly occupied sites.

In second-quantized Fock-space notation, lifting a one-body matrix $h_{pq}$ produces $\sum_{p,q} h_{pq}\, \hat c^\dagger_p \hat c_q$, so nearest-neighbour transfer lifts to $\sum_{i,\sigma} (\hat c^\dagger_{i+1,\sigma} \hat c_{i\sigma} + \hat c^\dagger_{i\sigma} \hat c_{i+1,\sigma})$ and double occupancy reads as $\sum_i \hat n_{i\uparrow} \hat n_{i\downarrow}$.

In [ ]:
def lifted_map(one_electron: OneElectron) -> Hamiltonian:
    """The derivation trace lifts `one_electron` (`Orbital+ <- Orbital+`) to act on each of the four
    occupied orbitals in turn (`[...] State+ <- State+`)."""
    # Joining with every complement removes one occupied orbital, wedging inserts its image with the
    # fermion exchange sign, and the trace pairs the map's input with the orbital removed.
    return (one_electron ^ (State & Complement)).trace(1, 3)                # [...] State+ <- State+


orbital_transfer = (next_orbitals * (orbitals | Orbital) + orbitals * (next_orbitals | Orbital)).sum(axis=0).sum(axis=0)   # [] Orbital+ <- Orbital+
ring_hopping = lifted_map(orbital_transfer)                                 # [] State+ <- State+
orbital_numbers = lifted_map(orbitals * (orbitals | Orbital))               # [4, 2] State+ <- State+
double_occupancy = orbital_numbers[:, 0](orbital_numbers[:, 1]).sum(axis=0)   # [] State+ <- State+

# Spin up on the first three sites and spin down on the fourth, alongside two doubly occupied sites.
initial_state = mv.a ^ mv.c ^ mv.e ^ mv.h                                   # [] State+
paired_state = mv.a ^ mv.b ^ mv.c ^ mv.d                                    # [] State+

In [ ]:
print("doubly occupied sites in initial_state and paired_state:",
      initial_state.reverse().scalar_product(double_occupancy(initial_state)).to_array(),
      paired_state.reverse().scalar_product(double_occupancy(paired_state)).to_array())

## Spin exchange from charge motion

Combining hopping with on-site repulsion gives `-hopping_strength * ring_hopping + repulsions * double_occupancy` as a symmetric linear map on 4-vectors. Placing one electron on each of the four sites leaves sixteen spin configurations with zero double occupancy, and increasing the repulsion separates those sixteen eigenstates from states with doubly occupied sites. Virtual hops to a neighbour and back lower the energy of antiparallel neighbours at rate `4 * hopping_strength**2 / repulsions`, splitting the sixteen spin levels into excitations at one, two, and three times that exchange rate above the singlet ground state.

In spin-operator notation, second-order degenerate perturbation theory projects onto the singly occupied subspace to give the antiferromagnetic Heisenberg ring $J \sum_{i=1}^{4} \mathbf S_i \cdot \mathbf S_{i+1}$ with $J = 4t^2 / U$, whose excitation energies $\tfrac{J}{2}[S(S+1) - S_A(S_A+1) - S_B(S_B+1)] + 2J$ follow from the total spin $S$ and the sublattice pair spins $S_A, S_B$.

In [ ]:
hopping_strength = 1.0
n_samples = 157
n_spin_levels = 16
repulsion_sweep = hopping_strength * np.linspace(1.0, 40.0, n_samples)
heisenberg_levels = np.array([1.0, 2.0, 3.0])

In [ ]:
def exchange_rate(hopping_strength: float, repulsions: np.ndarray) -> np.ndarray:
    """Virtual neighbour hops at strong on-site `repulsions` (`[...]`) couple adjacent spins at rate
    `4 * hopping_strength**2 / repulsions` (`[...]`)."""
    return 4 * hopping_strength**2 / repulsions


def spin_excitations(hopping_strength: float, repulsions: np.ndarray, n_spin_levels: int) -> Scalar:
    """Diagonalizing the Hubbard ring across `repulsions` (`[...]`) gives the lowest `n_spin_levels`
    excitation energies in units of the spin exchange (`[..., n_spin_levels] Scalar+`)."""
    hamiltonians = -hopping_strength * ring_hopping + repulsions * double_occupancy   # [...] State+ <- State+
    energies = hamiltonians.eigvalsh()                                                # [..., 70] Scalar+
    exchange = exchange_rate(hopping_strength, repulsions)                            # [...]
    return (energies[..., :n_spin_levels] - energies[..., :1]) / exchange[..., None]  # [..., n_spin_levels] Scalar+


excitations = spin_excitations(hopping_strength, repulsion_sweep, n_spin_levels)      # [n_samples, n_spin_levels] Scalar+
save_figure(render.levels_figure(repulsion_sweep / hopping_strength, excitations, heisenberg_levels), "hubbard_ring_levels");

Purple curves trace the fifteen lowest non-zero excitations above the ground state in units of the spin exchange as repulsion grows relative to hopping, bundling into multiplets of three, seven, and five levels along the grey dashed Heisenberg lines at one, two, and three.

In [ ]:
print("sixteen lowest levels in exchange units at maximum repulsion:\n", excitations[-1].to_array())

## Flipped spin walking around the ring

Starting from spin up on three sites and spin down on the fourth, each eigenstate of the Hamiltonian rotates at its own energy so the evolved state splits into real cosine and sine 4-vector parts without introducing an external imaginary unit. Evaluating each orbital number map on both parts gives the spin-up and spin-down electron counts around the ring over one cycle of the exchange phase. Because the four-site Heisenberg ring has evenly spaced spin excitation levels, interference between the two paths around the ring produces the perfect state transfer analysed by Christandl, Datta, Ekert, and Landahl (Physical Review Letters 92, 187902, 2004) in the strong-repulsion limit.

In spin-operator notation, unitary evolution $e^{-i\hat H\tau/\hbar}|\psi(0)\rangle$ on the four-site Heisenberg ring transfers a single flipped spin to the antipodal site with probability $\sin^4(J\tau / 2)$, reaching unity at $J\tau = \pi$.

In [ ]:
n_cases = 3
repulsions = hopping_strength * np.array([4.0, 8.0, 20.0])
n_frames = 189
frame_duration_ms = 40
exchange_phases = np.linspace(0.0, 2 * np.pi, n_frames)

In [ ]:
def spin_walk(hopping_strength: float, repulsions: np.ndarray, initial_state: State,
              exchange_phases: np.ndarray) -> Iterator[Scalar]:
    """Yields the electron occupation of every site and spin orbital (`[..., 4, 2] Scalar+`) across
    `exchange_phases` (`[n_frames]`) as a flipped spin walks around the ring."""
    hamiltonians = -hopping_strength * ring_hopping + repulsions * double_occupancy   # [...] State+ <- State+
    energies, modes = hamiltonians.eigh()                                             # [..., 70] Scalar+, State+
    shares = modes * modes.reverse().scalar_product(initial_state)                    # [..., 70] State+
    exchange = exchange_rate(hopping_strength, repulsions)                            # [...]
    for phase in exchange_phases:
        # Each eigenstate rotates at its own energy, splitting the evolved state into real cosine and sine parts.
        angles = energies * (phase / exchange)[..., None]                             # [..., 70] Scalar+
        cosine_part = (shares * angles.cos()).sum(axis=-1)[..., None, None]           # [..., 1, 1] State+
        sine_part = (shares * angles.sin()).sum(axis=-1)[..., None, None]             # [..., 1, 1] State+
        yield (cosine_part.reverse().scalar_product(orbital_numbers(cosine_part))
               + sine_part.reverse().scalar_product(orbital_numbers(sine_part)))      # [..., 4, 2] Scalar+


occupations = stack(show_states(
    spin_walk(hopping_strength, repulsions, initial_state, exchange_phases),
    render.ring_frames,
    repulsions / hopping_strength,
    name="hubbard_ring_walk",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
))                                                                                    # [n_frames, n_cases, 4, 2] Scalar+

Red upward bars show spin-up electron occupation and blue downward bars show spin-down occupation at each of the four sites around the ring. Starting at the top-left site, the flipped down spin splits symmetrically across both neighbours, refocuses on the opposite bottom-right site halfway through the exchange cycle, and returns to its starting site, arriving nearly whole at strong repulsion while charge fluctuations blur the transfer at weaker repulsion.

In [ ]:
print("opposite-site spin-down occupation halfway through the cycle:", occupations[n_frames // 2, :, 1, 1].to_array())

In [ ]:
# checks
# The derivation lift matches applying orbital_transfer to each occupied orbital in turn, double_occupancy
# counts doubly occupied sites, total electron number stays 4, and strong repulsion approaches the Heisenberg ring.
derivation_sum = ((orbital_transfer(mv.a) ^ mv.c ^ mv.e ^ mv.h)
                  + (mv.a ^ orbital_transfer(mv.c) ^ mv.e ^ mv.h)
                  + (mv.a ^ mv.c ^ orbital_transfer(mv.e) ^ mv.h)
                  + (mv.a ^ mv.c ^ mv.e ^ orbital_transfer(mv.h)))                    # [] State+
assert (ring_hopping(initial_state) - derivation_sum).scalar_norm_squared() < 1e-9
assert (double_occupancy(paired_state) - 2 * paired_state).scalar_norm_squared() < 1e-9
assert ((excitations[-1, 1:4] - heisenberg_levels[0]).abs() < 0.05).all()
assert ((excitations[-1, 4:11] - heisenberg_levels[1]).abs() < 0.05).all()
assert ((excitations[-1, 11:16] - heisenberg_levels[2]).abs() < 0.05).all()
assert ((occupations.sum(axis=(-1, -2)) - 4).abs() < 1e-9).all()
assert ((occupations[:, -1, 1, 1] - np.sin(exchange_phases / 2) ** 4).abs() < 0.05).all()